In [ ]:
import json
import logging
import pandas as pd
from kafka import KafkaConsumer
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
import os
from dotenv import load_dotenv

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler("pipeline.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

load_dotenv("../.env")

DB_URL = URL.create(
    "postgresql+psycopg2",
    username="postgres",
    password=os.getenv("SUPABASE_PASSWORD"),
    host="db.vqssaigndessrttplwib.supabase.co",
    port=5432,
    database="postgres"
)

engine = create_engine(DB_URL)

consumer = KafkaConsumer(
    "weather_stream",
    bootstrap_servers="localhost:9092",
    value_deserializer=lambda v: json.loads(v.decode("utf-8")),
    auto_offset_reset="latest"
)

logger.info("Listening to weather_stream...")

for message in consumer:
    data = message.value
    df = pd.DataFrame([data])
    df.to_sql("weather_stream_raw", engine, if_exists="append", index=False)
    logger.info(f"Stored: {data}")

2026-05-08 22:29:51,868 - INFO - <BrokerConnection client_id=kafka-python-2.3.1, node_id=bootstrap-0 host=localhost:9092 <connecting> [IPv6 ('::1', 9092, 0, 0)]>: connecting to localhost:9092 [('::1', 9092, 0, 0) IPv6]
2026-05-08 22:29:51,876 - INFO - <BrokerConnection client_id=kafka-python-2.3.1, node_id=bootstrap-0 host=localhost:9092 <checking_api_versions_recv> [IPv6 ('::1', 9092, 0, 0)]>: Broker version identified as 2.6
2026-05-08 22:29:51,877 - INFO - <BrokerConnection client_id=kafka-python-2.3.1, node_id=bootstrap-0 host=localhost:9092 <connected> [IPv6 ('::1', 9092, 0, 0)]>: Connection complete.
2026-05-08 22:29:51,877 - WARNING - group_id is None: disabling auto-commit.
2026-05-08 22:29:51,878 - INFO - Updating subscribed topics to: ('weather_stream',)
2026-05-08 22:29:51,878 - INFO - Listening to weather_stream...
2026-05-08 22:29:51,882 - INFO - Updated partition assignment: [TopicPartition(topic='weather_stream', partition=0)]
2026-05-08 22:29:51,884 - INFO - <BrokerConn